# 05 — Ablation Study
Goal: prove that the **hybrid coupling** is what drives the gain, not just one of the components.

Eight conditions evaluated on the **same NSL-KDD test set** with the **same threshold-tuning protocol**:

| # | Condition | What it isolates |
|---|---|---|
| 1 | **AE only** (recon-error threshold) | Pure DL one-class baseline (Phase 2) |
| 2 | **Isolation Forest on raw features** | Pure ML one-class baseline (Phase 1) |
| 3 | **Deep Isolation Forest** (Model A) | DL feature extractor + ML decision rule |
| 4 | **XGBoost on raw features only** | Pure ML supervised baseline |
| 5 | **XGBoost on raw + AE latent** (no residuals) | Adds DL representation, removes residual signal |
| 6 | **XGBoost on raw + residuals** (no latent) | Adds DL error signal, removes latent representation |
| 7 | **XGBoost on raw + latent + residuals** (full Model C, no gate) | Full hybrid feature set |
| 8 | **Cascaded AE + XGBoost** (full Model C, gated) | Adds Stage-1 AE gate on top of full hybrid |

All eight rows are emitted into `results/ablation_phase3.csv`. The diagnostic narrative — *which* component contributed *what* — is written into `results/report.md` by notebook 06.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest

sys.path.insert(0, os.path.abspath('.'))
from utils.config import RESULTS_DIR, MODELS_DIR, FIGURES_DIR, DEVICE, SEED
from utils.data_loader import load_splits
from utils.hybrid_models import (
    AutoencoderSkip, encode_dataset, recon_error,
    DeepIsolationForest, CascadedAE_XGB,
)
from utils.evaluation import find_threshold, binary_metrics, plot_metric_bars

torch.manual_seed(SEED); np.random.seed(SEED)
sns.set_style('whitegrid')

In [ ]:
splits = load_splits(os.path.join(RESULTS_DIR, 'processed_data.npz'))
X_tr   = splits['X_train_normal_scaled']
X_vm   = splits['X_val_mixed_scaled']
y_vm   = splits['y_val_mixed']
X_clf  = splits['X_clf_train_scaled']
y_clf  = splits['y_clf_train_multi']
X_te   = splits['X_test_scaled']
y_te   = splits['y_test']
input_dim = splits['input_dim']

ae = AutoencoderSkip(dim=input_dim, latent_dim=32)
ae.load_state_dict(torch.load(os.path.join(MODELS_DIR, 'phase3_ae.pth'),
                              map_location=DEVICE))
ae.to(DEVICE).eval()
print('AE loaded.')

In [ ]:
rows = []

def threshold_and_score(scores_val, scores_test, name):
    thr, _ = find_threshold(scores_val, y_vm)
    pred = (scores_test > thr).astype(int)
    return binary_metrics(y_te, pred, scores_test, name=name)

### Condition 1 — AE only

In [ ]:
_, recon_val  = encode_dataset(ae, X_vm, DEVICE)
_, recon_test = encode_dataset(ae, X_te, DEVICE)
ae_val   = recon_error(X_vm, recon_val)
ae_test  = recon_error(X_te, recon_test)
rows.append(threshold_and_score(ae_val, ae_test, '1. AE only'))

### Condition 2 — Isolation Forest on raw features

In [ ]:
raw_if = IsolationForest(n_estimators=200, contamination=0.05,
                         random_state=SEED, n_jobs=-1).fit(X_tr)
if_val  = -raw_if.score_samples(X_vm)
if_test = -raw_if.score_samples(X_te)
rows.append(threshold_and_score(if_val, if_test, '2. Isolation Forest (raw)'))

### Condition 3 — Deep Isolation Forest (Model A)

In [ ]:
dif = DeepIsolationForest(ae, DEVICE, n_estimators=200, contamination=0.05).fit(X_tr)
dif_val  = dif.score(X_vm)
dif_test = dif.score(X_te)
rows.append(threshold_and_score(dif_val, dif_test, '3. Deep IF (Model A)'))

### Condition 4 — XGBoost on raw features only (pure ML supervised baseline)

In [ ]:
cax_raw = CascadedAE_XGB(ae, DEVICE,
                         use_raw=True, use_latent=False, use_residuals=False)
cax_raw.fit(X_clf, y_clf)
scores_raw = cax_raw.predict_binary_score(X_te)
pred_raw   = (cax_raw.predict_multi(X_te) != 0).astype(int)
rows.append(binary_metrics(y_te, pred_raw, scores_raw,
                           name='4. XGBoost (raw only)'))

### Condition 5 — XGBoost on raw + AE latent (no residuals)

In [ ]:
cax_rz = CascadedAE_XGB(ae, DEVICE,
                        use_raw=True, use_latent=True, use_residuals=False)
cax_rz.fit(X_clf, y_clf)
scores_rz = cax_rz.predict_binary_score(X_te)
pred_rz   = (cax_rz.predict_multi(X_te) != 0).astype(int)
rows.append(binary_metrics(y_te, pred_rz, scores_rz,
                           name='5. XGBoost (raw + latent z)'))

### Condition 6 — XGBoost on raw + residuals (no latent)

In [ ]:
cax_rr = CascadedAE_XGB(ae, DEVICE,
                        use_raw=True, use_latent=False, use_residuals=True)
cax_rr.fit(X_clf, y_clf)
scores_rr = cax_rr.predict_binary_score(X_te)
pred_rr   = (cax_rr.predict_multi(X_te) != 0).astype(int)
rows.append(binary_metrics(y_te, pred_rr, scores_rr,
                           name='6. XGBoost (raw + residuals)'))

### Condition 7 — Full Model C (raw + latent + residuals, no gate)

In [ ]:
cax_full = CascadedAE_XGB(ae, DEVICE,
                          use_raw=True, use_latent=True, use_residuals=True)
cax_full.fit(X_clf, y_clf)
scores_full = cax_full.predict_binary_score(X_te)
pred_full   = (cax_full.predict_multi(X_te) != 0).astype(int)
rows.append(binary_metrics(y_te, pred_full, scores_full,
                           name='7. Model C (raw + z + residuals)'))

### Condition 8 — Full Model C with the AE Stage-1 gate enabled

In [ ]:
thr_gate, _ = find_threshold(ae_val, y_vm)
cax_full.detector_threshold = thr_gate
pred_gated   = cax_full.predict_cascade(X_te)
pred_gated_b = (pred_gated != 0).astype(int)
# Re-use the soft scores from Condition 7 for AUC continuity.
rows.append(binary_metrics(y_te, pred_gated_b, scores_full,
                           name='8. Model C + AE gate (cascade)'))

## Compile the table & diagnostic deltas

In [ ]:
df = pd.DataFrame(rows)
df = df[['model', 'accuracy', 'precision', 'recall', 'f1', 'auc']]
df.to_csv(os.path.join(RESULTS_DIR, 'ablation_phase3.csv'), index=False)
df.style.format({c: '{:.4f}' for c in df.columns if c != 'model'})

In [ ]:
# Diagnostic deltas — what does each component add on top of the previous one?
f1 = dict(zip(df['model'], df['f1']))
deltas = pd.DataFrame([
    {'change': 'Latent representation (5 vs 4)',
     'f1_delta': f1['5. XGBoost (raw + latent z)']    - f1['4. XGBoost (raw only)']},
    {'change': 'Residual signal (6 vs 4)',
     'f1_delta': f1['6. XGBoost (raw + residuals)']   - f1['4. XGBoost (raw only)']},
    {'change': 'Latent + residuals together (7 vs 4)',
     'f1_delta': f1['7. Model C (raw + z + residuals)'] - f1['4. XGBoost (raw only)']},
    {'change': 'AE gate on top (8 vs 7)',
     'f1_delta': f1['8. Model C + AE gate (cascade)']  - f1['7. Model C (raw + z + residuals)']},
    {'change': 'Hybrid vs raw IF (3 vs 2)',
     'f1_delta': f1['3. Deep IF (Model A)']            - f1['2. Isolation Forest (raw)']},
    {'change': 'Hybrid vs AE alone (3 vs 1)',
     'f1_delta': f1['3. Deep IF (Model A)']            - f1['1. AE only']},
])
deltas.to_csv(os.path.join(RESULTS_DIR, 'ablation_phase3_deltas.csv'), index=False)
deltas.style.format({'f1_delta': '{:+.4f}'})

In [ ]:
plot_metric_bars(
    df, metric='f1', title='Phase 3 ablation — F1 on test set',
    path=os.path.join(FIGURES_DIR, 'phase3_ablation_f1.png'),
)
plot_metric_bars(
    df, metric='auc', title='Phase 3 ablation — ROC-AUC on test set',
    path=os.path.join(FIGURES_DIR, 'phase3_ablation_auc.png'),
    color='#9b59b6',
)